# Logos Pretraining - Colab GPU

End-to-end pretraining of a single **Logos** transformer using the current KDA/SWA/CSA/HCA architecture.

**Architecture**
- KDA recurrent/chunkwise attention.
- SWA local sliding-window attention.
- CSA learned 4-token compression plus sparse top-k global recall.
- HCA learned heavier compression plus dense compressed global recall.
- Entry / Body / Exit looped-depth structure with Block Attention Residuals.
- MoE FFNs tuned for about 1.0B stored parameters, about 330M unique active parameters, and about 546M loop-counted active parameter-equivalent per token.

**Scaling target**
- Chinchilla-style budget: 20 tokens per active parameter.
- Loop-counted active parameter-equivalent estimate: about 546M.
- Training budget: 11B tokens on FineWeb-Edu sample-100BT.


In [ ]:
!nvidia-smi


In [ ]:
import pathlib, subprocess
REPO_URL = 'https://github.com/Rorical/Logos.git'
REPO_PATH = pathlib.Path('/content/Logos')
if REPO_PATH.exists():
    subprocess.check_call(['git', '-C', str(REPO_PATH), 'fetch', '--all', '--prune'])
    subprocess.check_call(['git', '-C', str(REPO_PATH), 'reset', '--hard', 'origin/master'])
else:
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_PATH)])
%cd /content/Logos
!pip install -q -U uv
!uv sync --extra wandb


## Mount Google Drive

Checkpoints land in `MyDrive/logos-pretraining/gpu/` so they survive runtime restarts and auto-resume from the latest checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib

CHECKPOINT_DIR = '/content/drive/MyDrive/logos-pretraining/gpu'
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print('Checkpoints will go to', CHECKPOINT_DIR)


In [ ]:
import sys, torch
sys.path.insert(0, '/content/Logos')
from scripts.train import build_arg_parser, main
assert torch.cuda.is_available(), 'Switch this Colab runtime to a GPU.'
print('torch:', torch.__version__)
print('device:', torch.cuda.get_device_name(0))


In [ ]:
parser = build_arg_parser()
cli = [
    '--model', 'logos',
    '--streaming',
    '--dataset', 'HuggingFaceFW/fineweb-edu',
    '--dataset-config', 'sample-100BT',
    '--text-column', 'text',
    '--val-docs', '200',
    '--tiktoken-encoding', 'cl100k_base',
    '--total-tokens', '11B',
    '--batch-size', '8',
    '--max-length', '4096',
    '--log-every', '50',
    '--eval-every', '10000',
    '--save-every', '5000',
    '--sample-every', '20000',
    '--keep-last-n', '5',
    '--num-workers', '8',
    '--prefetch-factor', '8',
    '--bf16',
    '--compile',
    '--compile-mode', 'default',
    '--gradient-checkpointing',
    '--ckpt-granularity', 'per-block',
    '--diagnostic-every', '100',
    '--d-model', '1024',
    '--num-heads', '16',
    '--head-dim', '64',
    '--d-ff', '2730',
    '--num-entry-layers', '2',
    '--num-body-layers', '6',
    '--num-exit-layers', '2',
    '--num-loops', '3',
    '--chunk-size', '128',
    '--conv-size', '4',
    '--entry-attn-pattern', 'hca,kda',
    '--body-attn-pattern', 'hca,csa,kda,swa,csa,kda,csa,hca,kda,swa,csa,kda,csa,csa,kda,swa,hca,kda',
    '--exit-attn-pattern', 'csa,swa',
    '--csa-compression', '4',
    '--csa-top-k', '64',
    '--csa-indexer-heads', '4',
    '--csa-indexer-dim', '32',
    '--hca-compression', '128',
    '--swa-window', '256',
    '--swa-every', '4',
    '--swa-offset', '3',
    '--use-moe',
    '--num-shared-experts', '2',
    '--num-sparse-experts', '32',
    '--top-k', '4',
    '--entry-top-k', '8',
    '--exit-top-k', '8',
    '--expert-d-ff', '832',
    '--moe-diversity-factor', '0',
    '--bias-update-rate', '0.02',
    '--moe-log-every', '1000',
    '--lm-head-chunk-size', '4096',
    '--muon',
    '--muon-schedule-hyperparams',
    '--lr', '0.004',
    '--embed-lr', '0.1',
    '--muon-lr', '0.01',
    '--router-lr', '1e-3',
    '--router-warmup-steps', '3500',
    '--weight-decay', '0.1',
    '--grad-clip', '1.0',
    '--scheduler', 'wsd',
    '--warmup-steps', '3500',
    '--decay-steps', '70000',
    '--decay-frac', '0.2',
    '--opt-state-log-every', '1000',
    '--ema-decay', '0.0',
    '--sample-prompt', 'In a recent study, researchers found that',
    '--sample-max-tokens', '80',
    '--sample-temperature', '0.8',
    '--save-dir', CHECKPOINT_DIR,
    '--seed', '42',
    '--wandb',
    '--wandb-project', 'logos-pretrain',
    '--wandb-mode', 'offline',
]
args = parser.parse_args(cli)
for k, v in sorted(vars(args).items()):
    print(f'{k:<28} {v}')


In [ ]:
main(args)
